# Проверка нового экспорта V2 на T4

Этот вспомогательный ноутбук предназначен для **нового** обучения.
Укажите путь к экспортированному ZIP в следующей ячейке.
Для сданных весов шага 111275 используйте `notebooks/inference_colab_t4.ipynb`:
он дополнительно сравнивает новые ответы с отправленной посылкой.


In [ ]:
import sys
import os
import json
import subprocess
import zipfile
import importlib.util
from pathlib import Path

ARCHIVE = Path('/content/t4_step111275.zip')  # Укажите фактическое имя; в Kaggle — путь /kaggle/input/...
WORK = Path('/content/hw2_v2_t4') if Path('/content').exists() else Path('/kaggle/working/hw2_v2_t4')
assert ARCHIVE.is_file(), f'Сначала загрузите экспортированный ZIP и исправьте ARCHIVE: {ARCHIVE}'
WORK.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(ARCHIVE) as z:
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in z.namelist())
    z.extractall(WORK)
os.chdir(WORK)
print('Bundle:', WORK)


In [ ]:
if importlib.util.find_spec('tokenizers') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'tokenizers==0.22.2'], check=True)
import torch
from tokenizers import Tokenizer
print('Torch:', torch.__version__)
assert torch.cuda.is_available()
print('GPU:', torch.cuda.get_device_name(0))
assert 'T4' in torch.cuda.get_device_name(0), 'Замер на другом GPU не подтверждает условия ДЗ'


## Три полных запуска
Первый запуск также сохраняется. Максимум из трёх должен быть ≤180 секунд, желательно ≤150 секунд.
Прогрев и загрузка модели вынесены в отдельные поля отчёта. Не утверждайте соответствие по оценке времени на подвыборке.
Если возникает OOM, уменьшите BATCH_SIZE; если слишком медленно, сравните 16/32/64 без изменения max_new_tokens.

In [ ]:
BATCH_SIZE = 32
subprocess.run([sys.executable, 'hw2_infer_t4_v2.py', '--bundle', str(WORK), '--batch-size', str(BATCH_SIZE),
                '--max-new-tokens', '384', '--repeats', '3', '--output', 't4_results'], check=True)
report = json.loads((WORK/'t4_results/t4_benchmark.json').read_text())
print(json.dumps(report, indent=2))
assert report['within_180_seconds']


## Итоговые файлы
`submission.jsonl` содержит ровно 1919 объектов **только с ключом code**, в исходном порядке test.
`prediction_metadata.json` хранит EOS и row_id отдельно. Программы здесь не исполняются.
Отправка в Telegram выполняется отдельно после проверки отчёта.

In [ ]:
submission = WORK/'t4_results/submission.jsonl'
rows = [json.loads(s) for s in submission.read_text().splitlines()]
assert len(rows) == 1919 and all(set(r) == {'code'} for r in rows)
print(submission)
print(WORK/'t4_results/t4_benchmark.json')
if Path('/content').exists():
    from google.colab import files
    files.download(str(WORK/'t4_results/t4_benchmark.json'))
    files.download(str(submission))
else:
    from IPython.display import FileLink, display
    display(FileLink(str(submission)))
    display(FileLink(str(WORK/'t4_results/t4_benchmark.json')))
